# Data Insights

## Module and Data Imports

In [ ]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
load_dotenv(dotenv_path="../.env")

data_path = os.getenv("data_path")
accounts_raw_data_path = os.getenv("accounts_data_path")
closure_raw_data_path = os.getenv("closure_data_path")
marketing_raw_data_path = os.getenv("marketing_data_path")

NOTEBOOK_DIR = "../"

In [ ]:
accounts_df = pd.read_parquet(f'{NOTEBOOK_DIR}{data_path}/accounts.parquet')
closure_df = pd.read_parquet(f'{NOTEBOOK_DIR}{data_path}/closure.parquet')
marketing_df = pd.read_parquet(f'{NOTEBOOK_DIR}{data_path}/marketing.parquet')
accounts_grouped = pd.read_parquet(f'{NOTEBOOK_DIR}{data_path}/accounts_summary.parquet')
channels_value_total = pd.read_parquet(f'{NOTEBOOK_DIR}{data_path}/channels_total_value.parquet')
channels_overview = pd.read_parquet(f'{NOTEBOOK_DIR}{data_path}/channels_overview.parquet')

## Palettes and Styles

In [ ]:
channels_palette = sns.color_palette("Set2", 8)

## Graphs

In [ ]:
fig, ax = plt.subplots(figsize = (12,10), ncols=2, nrows=2)
sns.scatterplot(data = channels_overview, x='total_spend', y='total_customers', hue='channel', palette=channels_palette, ax=ax[0, 0])
ax[0, 0].set_title('Spend vs Total Customers by Channel')
sns.scatterplot(data = channels_overview, x='total_spend', y='total_revenue', hue='channel', palette=channels_palette, ax=ax[0, 1])
ax[0, 1].set_title('Spend vs Revenue by Channel')
sns.scatterplot(data = channels_overview, x='total_revenue', y='total_customers', hue='channel', palette=channels_palette, ax=ax[1, 0])
ax[1, 0].set_title('Revenue vs Total Customers by Channel')
ax[1, 1].axis('off')
handles, labels = ax[0, 0].get_legend_handles_labels()
ax[1, 1].legend(handles, labels, loc='lower right', title='Acquisition Channels', title_fontsize=26, prop={'size': 20})
ax[0, 0].get_legend().remove()
ax[0, 1].get_legend().remove()
ax[1, 0].get_legend().remove()
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize = (8,6))
sns.barplot(data = channels_value_total, x='acquisition_channel', y='churn_rate', hue='acquisition_channel', palette=channels_palette)
ax.set_title('Churn Rate by Channel')

current_channels = [tick.get_text() for tick in ax.get_xticklabels()]

# Create a mapped list of string labels based on the boolean column values
custom_labels = []
for channel in current_channels:
    is_promo = channels_value_total.loc[channel, 'most_joined_on_promo'] == True
    
    text_label = "More on promo" if is_promo else "Less on promo"
    custom_labels.append(text_label)

# Iteratively annotate each bar with the corresponding custom label
bars = [p for p in ax.patches if p.get_height() > 0]
for bar, label_text in zip(bars, custom_labels):
    height = bar.get_height()
    x_pos = bar.get_x() + bar.get_width() / 2.0
    ax.annotate(label_text, xy=(x_pos, height), xytext=(0, 5),
                textcoords="offset points", ha='center', va='bottom',
                fontsize=10, weight='bold')

plt.tight_layout()

# Sources
'''
Google Search AI - Used this to learn how to add custom labels to bar plots in Seaborn and Matplotlib.
https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.annotate.html
https://stackoverflow.com/questions/28931224/how-to-add-value-labels-on-a-bar-chart
'''

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

month_bins = [0, 6, 12, 18, 24, 36, 48]
bin_labels = ['0-6m', '6-12m', '12-18m', '18-24m', '24-36m', '36-48m']

accounts_grouped['tenure_bucket'] = pd.cut(
    accounts_grouped['tenure_months'], 
    bins=month_bins, 
    labels=bin_labels, 
    include_lowest=True
)

heatmap_matrix = pd.pivot_table(
    data=accounts_grouped,
    index='acquisition_channel',
    columns='tenure_bucket',
    values='LTV',
    aggfunc='mean'
)

fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(
    data=heatmap_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar_kws={'label': 'Average User LTV (£)'},
    ax=ax
)

ax.set_title('Average User Lifetime Value (LTV) by Channel and Tenure', fontsize=14, weight='bold')
ax.set_xlabel('Tenure Months')
ax.set_ylabel('Acquisition Channel')

plt.tight_layout()
plt.show()
